# YOLO Hyperparameter Tuning

This notebook demonstrates how to perform hyperparameter tuning for YOLO training using Optuna in a sequential manner. This approach prevents Out-Of-Memory (OOM) errors and ensures the best combination of parameters are identified for our rock cutting dataset.

In [1]:
import os
import gc
import torch
import optuna
from ultralytics import YOLO
from pathlib import Path
import random
import string

# Prevent memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

VERSION = "Batch3and4_YOLO"
DATASET_ROOT = Path("/home/praktikan/projects/DwiAnggara/Datasets") / VERSION
DATASET_YAML = DATASET_ROOT / "data.yaml"

YOLO_MODEL = "yolo26m-seg.pt"
IMG_SIZE = 640
BATCH_SIZE = 16

def generate_runner_name(prefix="Tuning"):
    suffix = ''.join(random.choices(string.ascii_letters + string.digits, k=11))
    return f"{prefix}_{suffix}"

TUNE_NAME = generate_runner_name()

In [ ]:
# Save for later use
import json
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"
os.environ["CUDA_LAUNCH_BLOCKING"] = "0"  # Keep async for performance

def run_optuna_tuning():
    print("\n" + "=" * 80)
    print("STARTING HYPERPARAMETER TUNING (OPTUNA)")
    print("=" * 80 + "\n")

    TUNE_EPOCHS = 30
    TUNE_ITERATIONS = 10
    
    # Safe batch sizing for memory constrained tuning
    tune_batch_size = max(2, BATCH_SIZE // 4)
    print(f"Using severely reduced batch_size: {tune_batch_size} (prevents OOM)")

    def objective(trial):
        print(f"\n🚀 Starting Trial {trial.number}")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # 🔧 Hyperparameter search space
        lr0 = trial.suggest_float("lr0", 1e-5, 1e-3, log=True)
        weight_decay = trial.suggest_float("weight_decay", 0.0, 0.001)
        hsv_s = trial.suggest_float("hsv_s", 0.3, 0.9)
        hsv_v = trial.suggest_float("hsv_v", 0.3, 0.9)
        cls_weight = trial.suggest_float("cls", 0.5, 2.0)
        box_weight = trial.suggest_float("box", 0.02, 0.1)
        mask_ratio = trial.suggest_int("mask_ratio", 1, 4)
        dropout = trial.suggest_float("dropout", 0.0, 0.5)

        model = YOLO(YOLO_MODEL)

        try:
            results = model.train(
                data=DATASET_YAML,
                epochs=TUNE_EPOCHS,
                imgsz=IMG_SIZE,
                patience=15,           # Stop if no improvement for 10 epochs
                save_period=-1,        # Don't save intermediate weights during tuning
                batch=tune_batch_size,
                optimizer="AdamW",
                lr0=lr0,
                weight_decay=weight_decay,
                hsv_s=hsv_s,
                hsv_v=hsv_v,
                cls=cls_weight,
                box=box_weight,
                mask_ratio=mask_ratio,
                dropout=dropout,
                project=str(DATASET_ROOT / "models" / TUNE_NAME),
                name=f"trial_{trial.number}",
                verbose=False,
                plots=False,
                save=False,
                workers=2,
                amp=True,              # ✅ Keep AMP (saves ~40% memory)
                cache=False,           # ❌ Disable image caching (saves RAM, slight speed tradeoff)
                rect=False,            # ❌ Disable rectangular training
                close_mosaic=0,        # ❌ Skip mosaic removal phase (saves memory in later epochs)
                # workers=0,             # ❌ Use main thread for data loading (reduces overhead)
                device=0 if torch.cuda.is_available() else 'cpu'
            )

            # ✅ FIX: Extract scalar metric (not array!)
            # Option 1: Use mAP50-95 (more robust than F1 for segmentation)
            if hasattr(results, 'box') and results.box is not None:
                # results.box.map50_95 is usually a scalar
                metric = float(results.box.map50_95) if hasattr(results.box, 'map50_95') else float(results.box.f1.mean())
            elif hasattr(results, 'seg') and results.seg is not None:
                metric = float(results.seg.map50_95) if hasattr(results.seg, 'map50_95') else float(results.seg.f1.mean())
            else:
                metric = 0.0
                
            # Ensure it's a Python float (not numpy scalar)
            return float(metric)

        except Exception as e:
            print(f"❌ Trial {trial.number} failed: {str(e)}")
            # Return a very low value instead of raising, so Optuna can continue
            return -999.0

        finally:
            # Cleanup to prevent memory leaks
            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    study = optuna.create_study(direction="maximize", study_name=TUNE_NAME)
    study.optimize(objective, n_trials=TUNE_ITERATIONS, gc_after_trial=True)
    
    print("\n" + "=" * 80)
    print("OPTUNA TUNING COMPLETED!")
    print("=" * 80)
    print("Best Hyperparameters:")
    for key, value in study.best_params.items():
        print(f"  - {key}: {value}")

    # After study.optimize() completes:
    if study.best_trial.value > 0:  # Ensure at least one successful trial
        print(f"\n🏆 Best Trial: #{study.best_trial.number}")
        print(f"📈 Best mAP50-95: {study.best_trial.value:.4f}")
        print("\n⚙️ Best Hyperparameters:")
        for k, v in study.best_params.items():
            print(f"   {k}: {v}")
        with open("best_params.json", "w") as f:
            json.dump(study.best_params, f, indent=2)
    else:
        print("⚠️ No successful trials - check logs for errors")

    return study.best_params

best_optuna_results = run_optuna_tuning()


[I 2026-04-23 14:44:23,201] A new study created in memory with name: Tuning_cERERzz7ob9



STARTING HYPERPARAMETER TUNING (OPTUNA)

Using severely reduced batch_size: 4 (prevents OOM)

🚀 Starting Trial 0
New https://pypi.org/project/ultralytics/8.4.41 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.35 🚀 Python-3.10.19 torch-2.12.0.dev20260407+cu128 CUDA:0 (NVIDIA GeForce RTX 5080, 15842MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=0.08240407659912044, cache=False, cfg=None, classes=None, close_mosaic=0, cls=0.9569433826318309, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/praktikan/projects/DwiAnggara/Datasets/Batch3and4_YOLO/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.24724119268990497, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7591131699705

[I 2026-04-23 14:45:45,320] Trial 0 finished with value: -999.0 and parameters: {'lr0': 4.7533690367096884e-05, 'weight_decay': 0.0005771058829008947, 'hsv_s': 0.7591131699705531, 'hsv_v': 0.7780638354382494, 'cls': 0.9569433826318309, 'box': 0.08240407659912044, 'mask_ratio': 2, 'dropout': 0.24724119268990497}. Best is trial 0 with value: -999.0.



🚀 Starting Trial 1
New https://pypi.org/project/ultralytics/8.4.41 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.35 🚀 Python-3.10.19 torch-2.12.0.dev20260407+cu128 CUDA:0 (NVIDIA GeForce RTX 5080, 15842MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=0.07421113370290316, cache=False, cfg=None, classes=None, close_mosaic=0, cls=0.9001024266319989, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/praktikan/projects/DwiAnggara/Datasets/Batch3and4_YOLO/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.40188088535267324, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.3628640171351869, hsv_v=0.3859474195107806, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_wid

[I 2026-04-23 14:48:43,860] Trial 1 finished with value: 0.2241523054335294 and parameters: {'lr0': 4.319708658982158e-05, 'weight_decay': 8.988809941991993e-05, 'hsv_s': 0.3628640171351869, 'hsv_v': 0.3859474195107806, 'cls': 0.9001024266319989, 'box': 0.07421113370290316, 'mask_ratio': 3, 'dropout': 0.40188088535267324}. Best is trial 1 with value: 0.2241523054335294.


...


🚀 Starting Trial 9
New https://pypi.org/project/ultralytics/8.4.41 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.35 🚀 Python-3.10.19 torch-2.12.0.dev20260407+cu128 CUDA:0 (NVIDIA GeForce RTX 5080, 15842MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=0.049863145550085544, cache=False, cfg=None, classes=None, close_mosaic=0, cls=1.4570660866204326, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/praktikan/projects/DwiAnggara/Datasets/Batch3and4_YOLO/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.10105700441687371, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.6664502566898645, hsv_v=0.31117542159848194, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_w

Exception in thread Thread-32 (_pin_memory_loop):
Traceback (most recent call last):
  File "/home/praktikan/miniconda3/envs/magangpdu-dwi/lib/python3.10/threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "/home/praktikan/miniconda3/envs/magangpdu-dwi/lib/python3.10/threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "/home/praktikan/miniconda3/envs/magangpdu-dwi/lib/python3.10/site-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
  File "/home/praktikan/miniconda3/envs/magangpdu-dwi/lib/python3.10/site-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
  File "/home/praktikan/miniconda3/envs/magangpdu-dwi/lib/python3.10/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
  File "/home/praktikan/miniconda3/envs/magangpdu-dwi/lib/python3.10/site-packages/torch/multiprocessing/reduction


OPTUNA TUNING COMPLETED!
Best Hyperparameters:
  - lr0: 0.0006508933606917804
  - weight_decay: 0.0005934088914896764
  - hsv_s: 0.5350605846365526
  - hsv_v: 0.5160084718515613
  - cls: 0.7782012676852937
  - box: 0.07996417668794903
  - mask_ratio: 3
  - dropout: 0.38775785777349714

🏆 Best Trial: #8
📈 Best F1-mean: 0.4476

⚙️ Best Hyperparameters:
   lr0: 0.0006508933606917804
   weight_decay: 0.0005934088914896764
   hsv_s: 0.5350605846365526
   hsv_v: 0.5160084718515613
   cls: 0.7782012676852937
   box: 0.07996417668794903
   mask_ratio: 3
   dropout: 0.38775785777349714
New https://pypi.org/project/ultralytics/8.4.41 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.35 🚀 Python-3.10.19 torch-2.12.0.dev20260407+cu128 CUDA:0 (NVIDIA GeForce RTX 5080, 15842MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=0.07996417668794903, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.778

KeyboardInterrupt: 

: 